In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
import joblib
import sys
import os

sys.path.append('..')
from src.features import build_features, FEATURE_COLS
app = pd.read_csv('../data/raw/application_train.csv')
bureau = pd.read_csv('../data/raw/bureau.csv')

df = build_features(app, bureau)

X = df[FEATURE_COLS]
y = df['TARGET']

c:\Users\Paul\credit-scoring\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Размер X: (307511, 29)
Размер y: (307511,)
Дефолт rate: 8.1%


In [ ]:
lgb_model = lgb.LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    lgb_model, X, y,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"LightGBM (дефолт):       {scores.mean():.4f} ± {scores.std():.4f}")

LogReg baseline:         0.7258
После feature eng:       0.7407
LightGBM (дефолт):       0.7568 ± 0.0046
Прирост vs feature eng:  +0.0161


In [ ]:
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    """
    Функция которую Optuna будет максимизировать.
    trial — объект который предлагает значения параметров.
    Возвращает ROC-AUC — чем выше тем лучше.
    """
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
    }
    
    model = lgb.LGBMClassifier(**params)
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nЛучший ROC-AUC: {study.best_value:.4f}")
print(f"Лучшие параметры:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Best trial: 18. Best value: 0.76107: 100%|██████████| 50/50 [8:01:33<00:00, 577.87s/it]    


Лучший ROC-AUC: 0.7611
Лучшие параметры:
  n_estimators: 790
  learning_rate: 0.042732320475249455
  num_leaves: 94
  max_depth: 5
  min_child_samples: 199
  subsample: 0.902701082869012
  colsample_bytree: 0.9408285784806656
  reg_alpha: 9.63883769942733
  reg_lambda: 2.1182972896884963e-05


In [ ]:
import joblib
best_params = {
    'n_estimators': 790,
    'learning_rate': 0.042732320475249455,
    'num_leaves': 94,
    'max_depth': 5,
    'min_child_samples': 199,
    'subsample': 0.902701082869012,
    'colsample_bytree': 0.9408285784806656,
    'reg_alpha': 9.63883769942733,
    'reg_lambda': 2.1182972896884963e-05,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X, y)
joblib.dump(final_model, '../data/processed/lgb_model.pkl')
loaded_model = joblib.load('../data/processed/lgb_model.pkl')
test_pred = loaded_model.predict_proba(X.iloc[:5])[:, 1]
print(np.round(test_pred, 4))

Модель сохранена → data/processed/lgb_model.pkl

Тест загрузки — скоры первых 5 клиентов:
[0.326  0.0438 0.0366 0.0444 0.0813]
